# Simple Single Lens (with added Complexities) simulation with MeepSAT

In the previous tutorial, we saw how we can define a simple single lens system and got trustable results produced with MeepSAT for the central pixel time reverse simulations. In this Tutorial we are going add a bit more complexity to that system and try to extract some useful quantitative and qualitative metric out from it. By the end of next two tutorials, you will learn how to:

- Define different types of absorbers used in CMB telescopes based on their taper type
- Define different shapes of the baffles and see how the shapes and flairs on the end affect the various quantitative metrics
- Doing both Time-Reverse and Time-Forward simulations followed by extracting the Poynting Metric giving us some key qualitative statements about the radiation dominated regions in the Telescope


As we did in the previous tutorial, we will import all the necessary packages and write similar code until defining the epsilon map! We will do that at once in the below cell.

In [ ]:
# Importing various Python libraries and MeepSAT modules
import sys
import os
import site
from pathlib import Path
import meep as mp
import numpy as np
import h5py
import matplotlib.pyplot as plt
import time
import json
import math

# Importing the MEEPSAT librarires
import meepsat.simulator as sim
import meepsat.meep_geometry as comp_meep
import meepsat.permittivity_components as comp_eps
import meepsat.stepfunctions as stepfunctions
import meepsat.json_to_script as json_to_script
import meepsat.field_analysis as mpsat_analysis
import meepsat.helpers as mpsat_helpers

# JSON file path representing mainly the different optical components parameters
json_file_path = 'auxilary_data/02_simple_single_lens_AddedComplexities_a/TRV/simple_single_lens_AddedComplexities_TRV_a.json'
data = mpsat_helpers.read_json(json_file_path)

# Some universal constants
c_mm_s = 299792458.0 * 1000.0  # Speed of light in mm/s (m/s -> mm/s)
# Frequency of the simulation
freq = 150.0  # Frequency in GHz
a = 1  # 1 meep unit = 1 mm  
wvl = c_mm_s / (freq * 1e9)  # Wavelength in mm
freq_meep = 1.0 / (wvl * a)
print("freq (meep units):", freq_meep)

# Edit the freq in the JSON file 
data["sources"]["source1"]["frequecy"] = freq_meep

# Savepath: For storing the output generated during the simulation
savepath = f'auxilary_data/02_simple_single_lens_AddedComplexities/TRV/output_files/{freq}GHz'
os.makedirs(savepath, exist_ok=True)
data["output"]["savepath"]["path"] = savepath

# Initialising MEEPSAT Simulation
cell_X, cell_Y, cell_Z = data["simulation"]['primary_params']['cell_size']['x'], data["simulation"]['primary_params']['cell_size']['y'], data["simulation"]['primary_params']['cell_size']['z'] # Cell Size without considering the PML thickness and its factor


# Initialize the simulation with the different parameters
mpsat_sim = sim.sim_init(sim_name= str(data["simulation"]["name"]),
                        cell_size= [cell_X, cell_Y, cell_Z], # [sx, sy, sz] in mm
                        smallest_freq= data["simulation"]['primary_params']['smallest_freq'], 
                        resolution= data["simulation"]['primary_params']['resolution'],
                        boundary_layer_type= data['boundary_layers']['boundary']['type'],
                        boundary_layer_size= data['boundary_layers']['boundary']['size'],
                        factor_dpml= data['boundary_layers']['boundary']['factor_dpml'])

# Checking resolution and PML thickness 
# This function will automatically check all the length scales and wavelength scales
data, mpsat_sim = sim.check_resolution_and_pml(
    data=data, 
    mpsat_sim=mpsat_sim,
    smallest_freq=data["simulation"]['primary_params']['smallest_freq'],
    highest_n=data["lenses"]["lens1"]["n_refr"]
)

# Print the simulation parameters
print("\nMEEPSAT SIMULATION PARAMETERS:")
mpsat_sim.print_simulation_parameters()

# Adding Source
source_list = []
exec(json_to_script.source_script(data))


# Boundary Layers
x_left_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.Low)
x_right_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.High)
y_down_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.Low)
y_up_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.High)

custom_boundary_layers = [x_left_boundary, x_right_boundary, y_down_boundary, y_up_boundary]

# Creating the epsilon map for the simulation cell
size_x, size_y, size_z = mpsat_sim.cell_size[0], mpsat_sim.cell_size[1], mpsat_sim.cell_size[2]
res = int(mpsat_sim.resolution)  # Ensure resolution is an integer
# Create the epsilon map: total size of the simulation cell in all the axis multiplied by the resolution + 1
epsilon_map = np.ones((int((size_x)*res+1), 
                       int((size_y)*res+1)), dtype = float)

`NOTE`: With the current version of MeepSAT, we need to add absorbers with an empty epsilon array. This is likely an feature issue which will be solved in the future versions (but it won't stop us in defining other geomtries OR cause any issues).

Now we will add the absorbers. For more details on how absorbers can be defined, you can follow this page: https://meepsat.readthedocs.io/en/latest/FEATURES/Absorbers/ 

In our case we have define Pyramidal taper absorber with base (p) 6 mm and base to height ratio to be 1.5 with $\epsilon = 5.4 + 0.8$ i 



In [ ]:
#~ Absorbers at the bottom of the wall
absorbers_down = comp_meep.Absorbers(
    # Absorber Dimensions:
    p=6,
    p_h_ratio=1.5,
    taper_type='Pyramidal',
    grid_size_sx=size_x,
    grid_size_sy=size_y,
    resolution=res,
    eps_array=epsilon_map,
    geometry_objects=[],
    # Impedance Parameters:
    z0=1,
    z1=1/math.sqrt(5.4),
    # Centre, Angle and orientation Parameters; 
    # They don't work if start and end points are given
    center_x_mm=0,
    center_y_mm=0,
    orientation="+y",
    angle_axis="x",
    # Substrate parameters
    substrate_thickness=[5, 
                         53],
    substrate_material=[None, # means use the absorber material as substrate
                        mp.perfect_electric_conductor],
    add_substrate = True,
    # Absorber Material parameters
    epsilon_r=5.4,
    epsilon_i=0.8,
    # material=None,
    material_type = 'narrow_bandwidth_absorption',
    freq=data["sources"]['source1']["frequecy"],
    # Placing absorbers between two points in a straight line
    start_point= (-51, -40),
    end_point= (170, -40),
    overall_factor = 1,
    plot_alpha=False,
    plot_profile=False,
    savepath=savepath,
    plot_mesh = False
)


# absorbers at the top of the wall
absorbers_up = comp_meep.Absorbers(
    # Absorber Dimensions:
    p=6,
    p_h_ratio=1.5,
    taper_type='Pyramidal',
    grid_size_sx=size_x,
    grid_size_sy=size_y,
    resolution=res,
    eps_array=epsilon_map,
    geometry_objects=[],
    # Impedance Parameters:
    z0=1,
    z1=1/math.sqrt(5.4),
    # Centre, Angle and orientation Parameters; 
    # They don't work if start and end points are given
    center_x_mm=0,
    center_y_mm=0,
    orientation="-y",
    angle_axis="x",
    # Substrate parameters
    substrate_thickness=[5, 
                         53],
    substrate_material=[None, # means use the absorber material as substrate
                        mp.perfect_electric_conductor],
    add_substrate = True,
    # Absorber Material parameters
    epsilon_r=5.4,
    epsilon_i=0.8,
    # material=None,
    material_type = 'narrow_bandwidth_absorption',
    freq=data["sources"]['source1']["frequecy"],
    # Placing absorbers between two points in a straight line
    start_point= (-51, 40),
    end_point= (170, 40),
    overall_factor = 1,
    plot_alpha=False,
    plot_profile=False,
    savepath=savepath,
    plot_mesh = False
)

mpsat_sim.meep_geometry.extend(absorbers_up.assemble())
mpsat_sim.meep_geometry.extend(absorbers_down.assemble())



Now let's add the forebaffles. Although we can generate complex forebaffle structure, for simplicity and for an initial start, we have considered the following parameters for the forebaffle geometry. 

In [ ]:
# Define the forebaffle geometry parameters 
forebaffle_height = 50 # mm
forebaffle_base = 86.60 # mm
forebaffle_hypotenuse = np.sqrt(forebaffle_height**2 + forebaffle_base**2) # mm
forebaffle_angle = np.degrees(np.arctan(forebaffle_height / forebaffle_base)) # degrees
forebaffle_thickness = 6
forebaffle_absorber_thick = 3
forebaffle_num_periods = 1
forebaffle_amplitude = 0

In [ ]:
import numpy as np
h = 700 
theta = 17 #deg

p = h*np.sin(np.radians(theta))
b= h*np.cos(np.radians(theta))
print("p:", p)
print("b:", b)
print("hypotenuse:", np.sqrt(p**2 + b**2))

Again for simplicity, we are defining a linear forebaffle by setting the following parameters. Very soon, we will see how we can change the shape of the baffles using these parameters:

``` 
scaling_factor=3
spline_degree=3
spline_smoothing=1
```

In [ ]:
# Bottom Forebaffle
x_start_bottom, y_start_bottom = -48.4, -41

fb_bottom = comp_meep.Forebaffle(
    mpsat_sim= mpsat_sim,
    epsilon_map= epsilon_map,
    angle_degrees=180+forebaffle_angle, #180-forebaffle_angle,
    x_vertex= x_start_bottom,
    y_vertex= y_start_bottom,
    hypotenuse= forebaffle_hypotenuse,
    material=mp.perfect_electric_conductor,  # ← Use this instead
    name="Bottom Forebaffle",
    shape='spline',
    num_periods=forebaffle_num_periods,
    amplitude=forebaffle_amplitude,
    no_of_points=500,
    scaling_factor=3,
    spline_degree=3,
    spline_smoothing=1,
    fb_thickness=forebaffle_thickness,
    add_absorber=True,
    absorber_side= 'above',
    absorber_epsilon_real=5.4,
    absorber_epsilon_imag=0.7,
    absorber_thickness=forebaffle_absorber_thick
)

fb_bottom_assembled = fb_bottom.assemble()
mpsat_sim.meep_geometry.extend(fb_bottom_assembled)

# Top Forebaffle
x_start_top, y_start_top = -48.4, 41

fb_top = comp_meep.Forebaffle(
    mpsat_sim= mpsat_sim,
    epsilon_map= epsilon_map,
    angle_degrees=180-forebaffle_angle, #180-forebaffle_angle,
    x_vertex= x_start_top,
    y_vertex= y_start_top,
    hypotenuse= forebaffle_hypotenuse,
    material=mp.perfect_electric_conductor,  # ← Use this instead
    name="Top Forebaffle",
    shape='spline',
    num_periods=forebaffle_num_periods,
    amplitude=forebaffle_amplitude,
    no_of_points=500,
    scaling_factor=3,
    spline_degree=3,
    spline_smoothing=1,
    fb_thickness=forebaffle_thickness,
    add_absorber=True,
    absorber_side= 'below',
    absorber_epsilon_real=5.4,
    absorber_epsilon_imag=0.7,
    absorber_thickness=forebaffle_absorber_thick
)

fb_top_assembled = fb_top.assemble()
mpsat_sim.meep_geometry.extend(fb_top_assembled)


Now let's just add the aperture and lenses using the approach we did in the previous Tutorial

In [ ]:
# Adding lens (if given)
exec(json_to_script.add_lens(data))

# Adding box slabs (if given)
exec(json_to_script.add_slab(data))

# Adding aperture (if given)
exec(json_to_script.add_aperture(data))

Now we are ready to define the Simulation object, extract the epsilon map and run the simulation. We will just borrow the code from previous Tutorial for this purpose

In [ ]:

# DEFINING THE SIMULATION OBJECT
# Mirror symmetry along y direction (x=0 plane)
symmetries = [mp.Mirror(mp.Y, phase=+1)] 

simulation = mp.Simulation(
    cell_size=mpsat_sim.cell,
    sources=source_list,
    resolution=mpsat_sim.resolution,
    boundary_layers=custom_boundary_layers,
    geometry=mpsat_sim.meep_geometry,
    epsilon_input_file = data["output"]["savepath"]["path"] + data["output"]["epsilon_h5_file"]["filename"] +"_epsilon_map" + ".h5",
    symmetries = symmetries,
    force_complex_fields= True)

simulation.use_output_directory(savepath)
# ---------------------------------------------

# Run simulation briefly to store the epsilon map
sim.plot_and_save_epsilon(
    simulation=simulation,
    savepath=savepath,
    filename_prefix="geometry_plot",
    epsilon_data_name="epsilon",
    size_x=size_x,
    size_y=size_y,
    vmin=0.5,
    vmax=3,
    cmap='viridis',
    figsize=(8, 4),
    dpi=300
)


# ---------------------------------------------
# Set the stepfunctions parameters
# Animation Parameters
stepfunctions.set_animation_params(anim_params= {'image_every': data["output"]["animation_options"]["image_every"], 
                                              'Nfps': data["output"]["animation_options"]["Nfps"], 
                                              'anim_file_name': savepath + "/"+ data["output"]["animation_options"]["movie_name"] + ".mp4"})

# Field Parameters
stepfunctions.set_field_params(field_params= {'size_x': size_x,
                                              'size_y': size_y,
                                              'savepath': savepath,
                                              'downsampling_factor_x': data["output"]["animation_options"]["downsample_x"],
                                              'downsampling_factor_y': data["output"]["animation_options"]["downsample_y"]})

# Runtime parameters
runtime_params = sim.calculate_runtime_parameters(
    source_freq=float(data["sources"]["source1"]["frequecy"]),
    total_time= 400,
    animation_timestep = data["output"]["animation_options"]["image_every"],
    points_per_period=10,
    extraction_offset=10
)

# ---------------------------------------------
simulation.run(mp.at_every(runtime_params["animation_timestep"], stepfunctions.Ez2_dB),
               mp.after_time(runtime_params["t0"], mp.at_every(runtime_params["dt"], stepfunctions.accumulate_efield_and_hfield)),
               mp.at_end(stepfunctions.save_animation),
               mp.at_end(stepfunctions.save_accumulated_fields),
               mp.at_end(stepfunctions.extract_xyzw),
               until = runtime_params["total_time"])

print("Simulation completed.")                                                 

# ---------------------------------------------

# Save the final edited JSON data
with open(data["output"]["savepath"]["path"] + data["simulation"]["name"] + "_simulation_data.json", "w") as f:
    json.dump(data, f, indent=2)
print(f"Simulation parameters saved to: {data['output']['savepath']['path']}{data['simulation']['name']}_simulation_data.json")



Let's do the same post simulation analysis like we did last time and visualize the S, E fields along with the far field beam pattern.

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

def load_fields(basepath, filename):
    # Construct the full path to the file
    filepath = os.path.join(basepath, filename)
    # Load the fields stored in npz files
    data = np.load(filepath)

    return data

basepath = os.path.join(savepath)
e_filename = 'efield_timeavg.npz'
h_filename = 'hfield_timeavg.npz'
xyzw_filename = 'xyzw.npz'
e_data = load_fields(basepath, e_filename)
h_data = load_fields(basepath, h_filename)
xyzw_data = load_fields(basepath, xyzw_filename)
print("E-field data keys:", e_data.files)
print("H-field data keys:", h_data.files)
print("XYZW data keys:", xyzw_data.files)


# TE component (Ez, Hx, Hy)
ez = e_data['ez_real'] + 1j * e_data['ez_imag']
hx = h_data['hx_real'] + 1j * h_data['hx_imag']
hy = h_data['hy_real'] + 1j * h_data['hy_imag']

# S vector components for TE mode
sx = -ez * np.conj(hy)
sy = ez * np.conj(hx)
sx_mag = np.abs(sx)
sy_mag = np.abs(sy)
s_total = np.sqrt(sx_mag**2 + sy_mag**2)
s_total_db = 10 * np.log10(s_total / np.max(s_total) + 1e-20)  # in dB
# efield magnitude
ez_power = np.abs(ez)**2
ez_power_db = 10 * np.log10(ez_power / np.max(ez_power) + 1e-20)  # in dB

def plot_field(simname, field_db, title, filename, xcoords, ycoords, freq,
               vmin=-40, vmax=0, 
               savepath= os.path.join('./../processed_data/'),
                show_plots=True):
    import matplotlib.pyplot as plt
    plt.style.use('default')
    
    plt.figure(figsize=(8, 6))
    plt.imshow(field_db.T, extent=(xcoords[0], xcoords[-1], ycoords[0], ycoords[-1]),
               origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
    plt.colorbar(label='dB')
    plt.title(title)
    plt.xlabel('x (mm)')
    plt.ylabel('y (mm)')

    if savepath:
        # Create directory with simname and frequency subdirectories
        save_dir = os.path.join(savepath, simname, f'{freq}GHz')
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, filename), dpi=300)
        # Save as a svg file as well for publication
        plt.savefig(os.path.join(save_dir, filename).replace('.png', '.svg'), dpi=300) 
        print(f"{title} plot saved to: {os.path.join(save_dir, filename)}")
    if show_plots:
        plt.show()

# E-field plot
plot_field(simname = 'simple_single_lens_AddedComplexities_TRV_a', 
            field_db=ez_power_db, 
            title='E-field Magnitude (dB)', 
            filename='ez_magnitude_db.png', 
            xcoords=xyzw_data['x_coords'], 
            ycoords=xyzw_data['y_coords'],
            freq=freq)
# S-vector plot
plot_field(simname = 'simple_single_lens_AddedComplexities_TRV_a', 
            field_db= s_total_db, 
            title='Poynting Vector Magnitude (dB)', 
            filename='s_magnitude_db.png', 
            xcoords=xyzw_data['x_coords'], 
            ycoords=xyzw_data['y_coords'],
            freq=freq)

Now let's extract the far field beam!

In [ ]:
import meepsat.field_analysis as analysis

freq_to_analyse = str(int(freq))
c = 2.998e+11  # Speed of light in mm/s
wvl_meep = c / (freq * 1e9)  # Wavelength in mm
print(f"Frequency to analyse: {freq_to_analyse} GHz, corresponding wavelength: {wvl_meep:.2f} mm")

zeropad = 15

# Aperture radius (in mm)
# Defines the physical extent of the aperture used for far-field calculation
aperture_radius = data["apertures"]["aperture1"]["diameter"]/2

# MeepSAT (Ez is the dominant component for TE mode in MeepSAT simulations)
# Extract a 1D slice at the aperture location (x=-60 mm)
x_index = (np.abs(xyzw_data['x_coords'] - (data["apertures"]["aperture1"]["pos_x"]))).argmin()
aperture_slice_meepsat = ez[x_index, :]  # Extract 1D slice at x=-60
aperture_slice_meepsat = np.abs(aperture_slice_meepsat)
# Set everything else to NaN except the values within the aperture diameter
aperture_slice_meepsat = np.where(np.abs(xyzw_data['y_coords']) <= aperture_radius, aperture_slice_meepsat, np.nan)
# Remove NaN indices from both the power and the corresponding y-coordinates
valid_indices = ~np.isnan(aperture_slice_meepsat)
aperture_slice_meepsat = aperture_slice_meepsat[valid_indices]
y_coords_meepsat = xyzw_data['y_coords'][valid_indices]
aperture_slice_meepsat = aperture_slice_meepsat/np.max(aperture_slice_meepsat)  # Normalize the power for better comparison
aperture_slice_meepsat_dB = 10 * np.log10(aperture_slice_meepsat + 1e-20)  # in dB


meepsat_farfield_dict = analysis.meepsat_farfield(efield= aperture_slice_meepsat,
             coords= y_coords_meepsat,
             wavelength= wvl_meep,
             resolution= mpsat_sim.resolution,
             zero_pad_beam=15,
             plot_label='MEEPSAT FDTD')

import matplotlib.pyplot as plt
plt.style.use('default')

# Plot the farfield patterns for comparison
plt.figure(figsize=(8, 6))
plt.plot(meepsat_farfield_dict['angle'], meepsat_farfield_dict['power_dB'], label= meepsat_farfield_dict['plot_label'])
plt.xlabel('Angle (degrees)')
plt.ylabel('Power (dB)')
plt.xlim(0, 20)
plt.ylim(-60,0)
plt.legend()

# Save the figure
save_dir = os.path.join('./../processed_data', 'simple_single_lens_ARC', f'{freq}GHz')
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'farfield_comparison.png'), dpi=300, bbox_inches='tight')

# Save as a svg file for better quality in publications
plt.savefig(os.path.join(save_dir, 'farfield_comparison.svg'), dpi=300, bbox_inches='tight')

plt.show()